# SAM RCE runs: isentropic streamfunction in $\theta_e$ and frozen $\theta_e$

Each experiment is one SAM run directory, with SAM's file names built from the run name `<case>_<caseid>`:
the 3D snapshots are `OUT_3D/<case>_<caseid>_<nproc>_<step>.nc` (concatenated in time), and the density
profile is `RHO` from the statistics file `OUT_STAT/<case>_<caseid>.nc`, at the time closest to each snapshot.

The streamfunction is computed with two vertical coordinates:

- $\theta_e$: `thermo.equivalent_potential_temperature` (liquid water reference, Emanuel 1994 with ice),
- $\theta_{ef}$: `thermo.frozen_equivalent_potential_temperature` (ice reference, Pauluis 2018).

Both are computed from the same snapshot, so each file is read only once. The conditional sums
keep the experiment and time dependence, as in `isentropic_streamfunction_frozen.ipynb`.

Use the **Python (iso)** kernel.

## Configuration

In [ ]:
from pathlib import Path

ROOT = Path("/scratch/op13/SAM-Torch")

# experiment ID -> (run directory, SAM run name <case>_<caseid>)
RUNS = {
    "RCE_REF": (ROOT / "RCE_REF", "RCE_REF_300K"),
    "RCE_REF_TKE_M2005": (ROOT / "RCE_REF_TKE_M2005", "RCE_REF_TKE_M2005_300K"),
}
FILES_3D = "OUT_3D/{prefix}_*.nc"   # only <prefix>_<nproc>_<step>.nc is used, so other runs' files are skipped
FILE_STAT = "OUT_STAT/{prefix}.nc"   # statistics file with RHO(time, z); set to None to compute the density
RHO_VAR = "RHO"

# SAM 6.11 3D snapshot output
#   W (m/s), TABS (K), QV, QN (g/kg), p(z) (mb), PP (Pa)
#   QI (g/kg): cloud ice, written by the M2005 microphysics. If present, liquid = QN - QI and ice = QI;
#   otherwise (SAM1MOM) QN is split with SAM's temperature ramp between T_ICE and T_LIQ.
VAR = dict(w="W", T="TABS", p="p", pp="PP", qv="QV", qn="QN", qi="QI")
Z_DIM = "z"
TIME_DIM = "time"
T_ICE, T_LIQ = 253.16, 273.16   # SAM1MOM partition of QN into ice and liquid
QN_CLOUD = 0.01   # g/kg: a grid point is cloudy where QN > QN_CLOUD (SAM's cloud-fraction threshold)

# Snapshots to process in each run (counted over all its files), e.g. slice(-48, None, 4) for a quick test
TIME_SLICE = slice(None)

# Bins (K) for each coordinate. The frozen theta_e is higher than theta_e for moist air
# (vapor carries the latent heat of sublimation), so check the ranges printed below and adjust.
BINS = {
    "theta_e":  dict(th_min=300.0, th_max=370.0, nbin=141),
    "theta_ef": dict(th_min=300.0, th_max=390.0, nbin=181),
}
LABEL = {"theta_e": r"$\theta_e$", "theta_ef": r"$\theta_{ef}$ (frozen)"}

In [ ]:
import re
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso
from isentropic import thermo

def run_files(run, prefix):
    """3D files of one run, sorted by name (i.e. by time step), and its reference profile file (or None)."""
    files = sorted(f for f in run.glob(FILES_3D.format(prefix=prefix))
                   if re.fullmatch(rf"{re.escape(prefix)}_\d+_\d+\.nc", f.name))
    if not FILE_STAT:
        return files, None
    stat = run / FILE_STAT.format(prefix=prefix)
    if not stat.exists():   # look for it anywhere in the run directory
        stat = next(iter(sorted(run.rglob(Path(FILE_STAT.format(prefix=prefix)).name))), None)
    return files, stat


for exp, (run, prefix) in RUNS.items():
    files, stat = run_files(run, prefix)
    print(f"{exp}: {len(files)} 3D files, reference profile: {stat}")
    assert files, f"no 3D files in {run}"

files0, stat0 = run_files(*RUNS[next(iter(RUNS))])
ds0 = xr.open_dataset(files0[0])
z = ds0[Z_DIM].values
ds0

## Thermodynamic fields at the first time of the first run

In [ ]:
def reference_density(stat):
    """RHO(time, z) from the reference profile file, or None."""
    if stat is None:
        return None
    with xr.open_dataset(stat) as st:
        if RHO_VAR not in st:
            print(f"{RHO_VAR} not in {stat.name}, computing the density instead")
            return None
        rho = st[RHO_VAR].load()
    assert rho.sizes[Z_DIM] == len(z), "reference profile and 3D output have different vertical levels"
    return rho


def load_time(ds, it, rho_ref=None):
    """Return rho*w, the coordinates {theta_e, theta_ef}, the density profile (nz) and the cloud mask
    (QN > QN_CLOUD) at time index it.

    The density is the reference profile at the closest time if rho_ref is given, otherwise the
    horizontal mean of the density computed from the snapshot."""
    snap = ds.isel({TIME_DIM: it})
    get = lambda name: snap[name].values.astype(float)
    T = get(VAR["T"])
    qv = np.maximum(get(VAR["qv"]) * 1e-3, 1e-10)           # g/kg -> kg/kg
    qn = np.maximum(get(VAR["qn"]) * 1e-3, 0.0)
    if VAR["qi"] in snap:
        qi = np.clip(get(VAR["qi"]) * 1e-3, 0.0, qn)
        ql = qn - qi
    else:
        liq_frac = np.clip((T - T_ICE) / (T_LIQ - T_ICE), 0.0, 1.0)
        ql, qi = qn * liq_frac, qn * (1.0 - liq_frac)
    p = get(VAR["p"])[:, None, None] * 100.0 + get(VAR["pp"])   # mb -> Pa, plus perturbation
    coords = {
        "theta_e": thermo.equivalent_potential_temperature(p, T, qv, ql, qi),
        "theta_ef": thermo.frozen_equivalent_potential_temperature(p, T, qv, ql, qi),
    }
    if rho_ref is None:
        rho = thermo.density(p, T, qv, ql, qi).mean((1, 2))    # density profile (anelastic)
    else:
        rho = rho_ref.sel({TIME_DIM: snap[TIME_DIM].values}, method="nearest").values
    cloudy = get(VAR["qn"]) > QN_CLOUD
    return rho[:, None, None] * get(VAR["w"]), coords, rho, cloudy


rho_ref0 = reference_density(stat0)
rhow, coords, rho, cloudy = load_time(ds0, 0, rho_ref0)
print(f"cloud fraction at t=0: {cloudy.mean():.3f} (max over levels {cloudy.mean((1, 2)).max():.3f})")
if rho_ref0 is not None:
    print("max relative difference, reference vs computed density:",
          np.max(np.abs(rho / load_time(ds0, 0)[2] - 1)))
for name, th in coords.items():
    print(f"{name:8s} range: {np.nanmin(th):.1f} - {np.nanmax(th):.1f} K   bins: {BINS[name]}")

## Conditional sums for each experiment and time
The result has one set of variables per coordinate: `MF_theta_e(experiment, time, z, theta_e)`,
`MF_theta_ef(experiment, time, z, theta_ef)`, and the same for `AREA` and `ATH` (sum of the coordinate itself).
`MF_CLD` and `AREA_CLD` are the same sums over the cloudy points only (QN > `QN_CLOUD`); the clear-air
sums are the total minus the cloudy ones.

In [ ]:
centers = {name: b["th_min"] + (b["th_max"] - b["th_min"]) / (b["nbin"] - 1) * np.arange(b["nbin"])
           for name, b in BINS.items()}
dth = {name: c[1] - c[0] for name, c in centers.items()}


def process_experiment(exp, run, prefix):
    """Conditional sums in both coordinates at every selected time of one run."""
    files, stat = run_files(run, prefix)
    rho_ref = reference_density(stat)
    # (file, time index in file) for every snapshot of the run, then select with TIME_SLICE
    # (a time present in several files is used once, from the last file)
    snaps = {}
    for f in files:
        with xr.open_dataset(f) as ds:
            for it, t in enumerate(ds[TIME_DIM].values):
                if t in snaps:
                    print(f"{exp}: time {t} in {snaps[t][0].name} and {f.name}, using {f.name}")
                snaps[t] = (f, it)
    snaps = [snaps[t] for t in sorted(snaps)][TIME_SLICE]
    out = {}
    for name, b in BINS.items():
        for v in ("MF", "AREA", "ATH", "MF_CLD", "AREA_CLD"):
            out[f"{v}_{name}"] = np.zeros((len(snaps), len(z), b["nbin"]))
    RHO = np.zeros((len(snaps), len(z)))
    time = np.zeros(len(snaps))
    for k, (f, it) in enumerate(snaps):
        with xr.open_dataset(f) as ds:
            time[k] = ds[TIME_DIM].values[it]
            rhow, coords, RHO[k], cloudy = load_time(ds, it, rho_ref)
        for name, th in coords.items():
            mask = ~np.isfinite(th)
            (mf, ath), area = iso.iso_multi_integral(np.stack([rhow, th]), th, mask, 1.0, **BINS[name])
            out[f"MF_{name}"][k], out[f"ATH_{name}"][k], out[f"AREA_{name}"][k] = mf, ath, area
            # cloudy points only; the clear-air sums are the total minus these
            (mf,), area = iso.iso_multi_integral(rhow[None], th, mask | ~cloudy, 1.0, **BINS[name])
            out[f"MF_CLD_{name}"][k], out[f"AREA_CLD_{name}"][k] = mf, area
        if k % 24 == 0:
            print(exp, "snapshot", k, "/", len(snaps), f.name)
    data = {key: (("time", "z", name), val) for name in BINS
            for key, val in out.items() if key.endswith("_" + name)}
    data["RHO"] = (("time", "z"), RHO)
    return xr.Dataset(data, coords={"time": time, "z": z, **centers}).expand_dims(experiment=[exp])


sums = xr.concat([process_experiment(exp, *r) for exp, r in RUNS.items()], dim="experiment", join="outer")
sums

## Streamfunctions
$\Psi$ is computed for every experiment and time in each coordinate, at the upper edge of each bin.
The cloudy and clear parts $\Psi_{cld}$ and $\Psi_{clr} = \Psi - \Psi_{cld}$ use the mass flux of the
cloudy (clear) points, divided by the total domain area, so that they add up to $\Psi$.
`CLD_FRAC(experiment, time, z)` is the cloud fraction profile.

In [ ]:
result = sums.copy()
for name in BINS:
    mf, area = sums[f"MF_{name}"].values, sums[f"AREA_{name}"].values
    ts = iso.two_stream(mf, area)
    result[f"PSI_{name}"] = (("experiment", "time", "z", name), iso.streamfunction(mf, area))
    result[f"PSI_{name}"].attrs.update(long_name=f"isentropic streamfunction in {name}", units="kg m-2 s-1",
                                       note=f"evaluated at {name} + d{name}/2")
    # cloudy and clear parts, both normalized by the total area so that PSI = PSI_CLD + PSI_CLR
    psi_cld = iso.streamfunction(sums[f"MF_CLD_{name}"].values, area)
    result[f"PSI_CLD_{name}"] = (("experiment", "time", "z", name), psi_cld)
    result[f"PSI_CLR_{name}"] = result[f"PSI_{name}"] - result[f"PSI_CLD_{name}"]
    result[f"PSI_CLD_{name}"].attrs.update(long_name=f"streamfunction of cloudy air in {name}", units="kg m-2 s-1")
    result[f"PSI_CLR_{name}"].attrs.update(long_name=f"streamfunction of clear air in {name}", units="kg m-2 s-1")
    result[f"CLD_FRAC_{name}"] = sums[f"AREA_CLD_{name}"].sum(name) / sums[f"AREA_{name}"].sum(name)
    result[f"MEAN_{name}"] = sums[f"ATH_{name}"].sum(name) / sums[f"AREA_{name}"].sum(name)
    result[f"M_UP_{name}"] = (("experiment", "time", "z"), ts["M_up"])
    result[f"M_DOWN_{name}"] = (("experiment", "time", "z"), ts["M_down"])
result.attrs.update(source=", ".join(str(r) for r, _ in RUNS.values()), **{f"d{name}": d for name, d in dth.items()})
result

## Comparison, one figure per experiment
Top: time-mean $\Psi$ in $\theta_e$ and in frozen $\theta_e$, with the same colour scale and the mean
profile of each coordinate. Bottom: the two-stream ascending and descending mass fluxes $M^\pm$, and the
overturning $\max_\theta |\Psi|$ in each coordinate. Both top panels use the same $\theta$ axis.

In [ ]:
# Time means: sum the conditional sums over time, then compute Psi
tm = {}
for name in BINS:
    mf = sums[f"MF_{name}"].sum("time", skipna=True).values
    area = sums[f"AREA_{name}"].sum("time", skipna=True).values
    ts = iso.two_stream(mf, area)
    tm[name] = dict(psi=iso.streamfunction(mf, area),
                    mean=sums[f"ATH_{name}"].sum("time", skipna=True).sum(name).values / area.sum(-1),
                    M_up=ts["M_up"], M_down=ts["M_down"])
lim = max(np.nanmax(np.abs(t["psi"])) for t in tm.values())
levels = np.linspace(-lim, lim, 20)
xlim = (min(b["th_min"] for b in BINS.values()), max(b["th_max"] for b in BINS.values()))

for e, exp in enumerate(sums.experiment.values):
    fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
    for ax, name in zip(axs[0], BINS):
        th = centers[name] + dth[name] / 2
        pc = ax.pcolormesh(th, z / 1e3, tm[name]["psi"][e], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
        ax.contour(th, z / 1e3, tm[name]["psi"][e], levels=levels, colors="k", linewidths=0.4)
        ax.plot(tm[name]["mean"][e], z / 1e3, "k", lw=2, label=f"mean {LABEL[name]}")
        ax.set_xlabel(LABEL[name] + " (K)")
        ax.set_xlim(xlim)
        ax.legend(loc="upper right")
    axs[0, 0].set_ylabel("z (km)")
    fig.colorbar(pc, ax=axs[0], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")

    ax = axs[1, 0]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(tm[name]["M_up"][e], z / 1e3, "C0", ls=ls, label=r"$M^+$, " + LABEL[name])
        ax.plot(-tm[name]["M_down"][e], z / 1e3, "C1", ls=ls, label=r"$-M^-$, " + LABEL[name])
    ax.set_xlabel(r"mass flux (kg m$^{-2}$ s$^{-1}$)")
    ax.set_ylabel("z (km)")
    ax.legend()

    ax = axs[1, 1]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(np.nanmax(np.abs(tm[name]["psi"][e]), axis=-1), z / 1e3, "k", ls=ls, label=LABEL[name])
    ax.set_xlabel(r"$\max_\theta |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
    ax.legend()

    nt = int(np.isfinite(sums.MF_theta_e.isel(experiment=e, z=0, theta_e=0)).sum())
    fig.suptitle(f"{exp}: time mean over {nt} snapshots")
    plt.show()

## $\theta_e$ versus frozen $\theta_e$
For each experiment: the time-mean, area-weighted profiles of the two coordinates, their difference, and the
joint distribution of $(\theta_e, \theta_{ef})$ over all grid points of the last processed snapshot.
For dry air the two are equal. The frozen coordinate uses ice as the reference state, so it is higher
wherever there is vapor or liquid water, which then also carry the latent heat of fusion.

In [ ]:
th_lim = (min(b["th_min"] for b in BINS.values()), max(b["th_max"] for b in BINS.values()))
hist_bins = np.linspace(*th_lim, 181)

for exp, (run, prefix) in RUNS.items():
    # time-mean profiles from the conditional sums (area weighted)
    mean = {name: (sums[f"ATH_{name}"].sel(experiment=exp).sum(("time", name))
                   / sums[f"AREA_{name}"].sel(experiment=exp).sum(("time", name))).values for name in BINS}

    # last processed snapshot
    files, stat = run_files(run, prefix)
    t_last = float(sums.time[np.isfinite(sums.MF_theta_e.sel(experiment=exp).isel(z=0, theta_e=0))][-1])
    for f in reversed(files):
        with xr.open_dataset(f) as ds:
            hit = np.flatnonzero(np.isclose(ds[TIME_DIM].values, t_last))
            if hit.size:
                _, coords, _, _ = load_time(ds, hit[0])
                break
    h, _, _ = np.histogram2d(coords["theta_e"].ravel(), coords["theta_ef"].ravel(), bins=hist_bins)

    fig, axs = plt.subplots(1, 3, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.3]})
    ax = axs[0]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(mean[name], z / 1e3, "k", ls=ls, label=LABEL[name])
    ax.set_xlabel("K")
    ax.set_ylabel("z (km)")
    ax.legend()
    ax.set_title("time-mean profiles")

    ax = axs[1]
    ax.plot(mean["theta_ef"] - mean["theta_e"], z / 1e3, "k")
    ax.set_xlabel(r"$\theta_{ef} - \theta_e$ (K)")
    ax.set_title("difference of the mean profiles")

    ax = axs[2]
    pc = ax.pcolormesh(hist_bins, hist_bins, np.where(h > 0, h, np.nan).T, norm="log", cmap="viridis")
    ax.plot(th_lim, th_lim, "r", lw=0.8, label="1:1")
    ax.set_xlabel(r"$\theta_e$ (K)")
    ax.set_ylabel(r"$\theta_{ef}$ (K)")
    ax.legend(loc="upper left")
    ax.set_title(f"joint distribution, t = {t_last:g} d")
    fig.colorbar(pc, ax=ax, label="number of grid points")

    fig.suptitle(exp)
    fig.tight_layout()
    plt.show()

## Cloudy and clear air, one figure per experiment
For each coordinate (rows): the time-mean $\Psi$, its cloudy and clear parts on the same colour scale,
and the fraction of cloudy points in each $(z, \theta)$ bin. The last two columns show the net upward mass
flux carried by cloudy and by clear air ($\sum_\theta$ MF / total area, which add up to the domain-mean
mass flux), and the cloud fraction profile.

In [ ]:
for e, exp in enumerate(sums.experiment.values):
    fig, axs = plt.subplots(len(BINS), 6, figsize=(24, 5 * len(BINS)), sharey=True, squeeze=False,
                            gridspec_kw={"width_ratios": [3, 3, 3, 3, 2, 1.2]}, layout="constrained")
    for row, name in zip(axs, BINS):
        s_e = sums.isel(experiment=e).sum("time", skipna=True)
        area = s_e[f"AREA_{name}"].values
        a_tot = area.sum(-1)
        psi = {"all": iso.streamfunction(s_e[f"MF_{name}"].values, area),
               "cloudy": iso.streamfunction(s_e[f"MF_CLD_{name}"].values, area)}
        psi["clear"] = psi["all"] - psi["cloudy"]
        lim = np.nanmax(np.abs(psi["all"]))
        th = centers[name] + dth[name] / 2
        for ax, (part, p) in zip(row[:3], psi.items()):
            pc = ax.pcolormesh(th, z / 1e3, p, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
            ax.contour(th, z / 1e3, p, levels=np.linspace(-lim, lim, 20), colors="k", linewidths=0.4)
            ax.set_title(rf"$\Psi$, {part}")
        fig.colorbar(pc, ax=row[2], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")

        with np.errstate(invalid="ignore", divide="ignore"):
            frac = np.where(area > 0, s_e[f"AREA_CLD_{name}"].values / area, np.nan)
        pc = row[3].pcolormesh(centers[name], z / 1e3, frac, cmap="Blues", vmin=0, vmax=1, shading="auto")
        row[3].set_title("fraction of cloudy points")
        fig.colorbar(pc, ax=row[3])
        for ax in row[:4]:
            ax.set_xlabel(LABEL[name] + " (K)")
            ax.set_xlim(BINS[name]["th_min"], BINS[name]["th_max"])
        row[0].set_ylabel("z (km)")

        ax = row[4]
        mf_cld = s_e[f"MF_CLD_{name}"].values.sum(-1) / a_tot
        mf_all = s_e[f"MF_{name}"].values.sum(-1) / a_tot
        ax.plot(mf_cld, z / 1e3, label="cloudy")
        ax.plot(mf_all - mf_cld, z / 1e3, label="clear")
        ax.plot(mf_all, z / 1e3, "k:", label="total")
        ax.set_xlabel(r"net mass flux (kg m$^{-2}$ s$^{-1}$)")
        ax.legend(loc="upper right")

        ax = row[5]
        ax.plot(s_e[f"AREA_CLD_{name}"].values.sum(-1) / a_tot, z / 1e3, "k")
        ax.set_xlabel("cloud fraction")

    fig.suptitle(f"{exp}: cloudy (QN > {QN_CLOUD} g/kg) and clear air, time mean")
    plt.show()

## Save

In [ ]:
result.to_netcdf("iso_streamfunction_sam_rce.nc")